# Image Manipulation and Analysis With `scikit-image`

In this lab, we'll use some of the features scikit-image offers for image manipulation and analysis.

Scikit-image works on numpy arrays, so a good knowledge of numpy and some ease with manipulating numpy arrays are needed to make the most of `scikit-image`.

In [ ]:
!wget -O la-havana.jpg https://upload.wikimedia.org/wikipedia/commons/thumb/7/74/AvMalecon-LaHabanaCuba-04735.jpg/1920px-AvMalecon-LaHabanaCuba-04735.jpg

from collections.abc import Iterable

import matplotlib.pyplot as plt
import numpy as np
import scipy
import skimage
from PIL import Image
from skimage import color, exposure, feature, transform
from skimage.color import rgb2gray
from skimage.feature import ORB
from skimage.filters import gaussian
from skimage.segmentation import active_contour


##################################################
# Display helper (ignore it) ###############
##################################################
def side_by_side(
  images: Iterable[Image.Image | np.ndarray],
  title: str | None = None,
  subtitles: Iterable[str] | None = None,
  dpi=150,
) -> None:
  fig, axes = plt.subplots(1, len(images), dpi=dpi)

  for i, (ax, im) in enumerate(zip(axes, images)):
    if type(im) is np.ndarray:
      mode = "gray" if len(im.shape) == 2 else "viridis"
    else:
      mode = "gray" if im.mode == "L" else "viridis"
    ax.imshow(im, cmap=mode)
    ax.axis("off")
    if subtitles:
      ax.set_title(subtitles[i], fontsize=8)

  if title:
    fig.suptitle(title, y=0.85)

  fig.subplots_adjust(wspace=0, hspace=0)
  fig.tight_layout()
  plt.show()


###################################

## Histogram
As you know, an image's histogram is simple information to get, which lets you manipulate an image to adjust its contrast.

As in the Pillow lab, start by loading an image into a numpy array. You'll find an image `la-havana.jpg` at the root of the current folder. Load it into a variable `im`.

In [ ]:
# Your code here

From this image, use the [`skimage.exposure.histogram`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.histogram) function to get and display the histogram of each channel.

In [ ]:
# Your code here

Use various histogram-based contrast correction functions:
 * [`skimage.exposure.equalize_hist`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_hist) to apply a linear correction
 * [`skimage.exposure.equalize_adapthist`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.equalize_adapthist) to apply an adaptive correction
 * [`skimage.exposure.rescale_intensity`](https://scikit-image.org/docs/stable/api/skimage.exposure.html#skimage.exposure.rescale_intensity) to apply a correction within a defined range of values (take the range `(30,220)` to start).

Display the resulting image and the new histogram.

In [ ]:
# Your code here

### Solution

In [ ]:
im = np.array(
  Image.open(
    "./la-havana.jpg",
  )
)

In [ ]:
def display_hist_gray(im: np.ndarray, f=lambda x: x):
  im_hist, im_val = exposure.histogram(im)
  plt.plot(im_val, f(im_hist))
  plt.show()


def display_hist_color(im: np.ndarray, f=lambda x: x):
  im_hist, im_val = exposure.histogram(im, channel_axis=2)
  for h, c in zip(im_hist, ("r", "g", "b")):
    plt.plot(im_val, f(h), color=c)
  plt.show()


def display_hist(im: np.ndarray, f=lambda x: x):
  if len(im.shape) == 3:
    display_hist_color(im, f)
  else:
    display_hist_gray(im, f)


def display_cumhist(im: np.ndarray):
  display_hist(im, np.cumsum)


display_hist(im)
display_cumhist(im)

In [ ]:
im_eq = exposure.equalize_hist(im)
side_by_side((im, im_eq), subtitles=("Original", "Equalized"))
display_hist(im_eq)
display_cumhist(im_eq)

In [ ]:
im_adapt = exposure.equalize_adapthist(im)
side_by_side((im, im_adapt), subtitles=("Original", "Adaptive equalization"))
display_hist(im_adapt)
display_cumhist(im_adapt)

In [ ]:
im_rescale = exposure.rescale_intensity(im, (30, 220))
side_by_side((im, im_rescale), subtitles=("Original", "Intensity rescaling"))
display_hist(im_rescale)
display_cumhist(im_rescale)

## Feature Detection

We'll use a method that detects features between two images.

To start, rotate your image by 45° to get a new image `im_rotate`, using [`skimage.transform.rotate`](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.rotate)

Use a feature detection model such as [`ORB`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.ORB) (**O**riented FAST and **R**otated **B**RIEF):
 * Create an object from the `ORB` class
 * Call your object's `detect_and_extract` method on your _**black and white**_ image
 * From your object, you can then get the `keypoints` and `descriptors` attributes

The `keypoints` are the coordinates of the features extracted from your image. Plot them on your image.

In [ ]:
# Your code here

### Solution

In [ ]:
im_rotate = transform.rotate(im, 45)
side_by_side((im, im_rotate), subtitles=("Original", "45° rotation"))

In [ ]:
# Create the model
feature_detector = ORB()

# Find the features on the B&W image
feature_detector.detect_and_extract(color.rgb2gray(im))

# Get the computed keypoints and descriptors
kp1 = feature_detector.keypoints
dsc1 = feature_detector.descriptors

# Display
plt.imshow(im)
plt.scatter(kp1[:, 1], kp1[:, 0])

In [ ]:
# Find the features on the B&W image
feature_detector.detect_and_extract(color.rgb2gray(im_rotate))

# Get the computed keypoints and descriptors
kp2 = feature_detector.keypoints
dsc2 = feature_detector.descriptors

# Display
plt.imshow(im_rotate)
plt.scatter(kp2[:, 1], kp2[:, 0])

## Feature Matching

Use the [`skimage.feature.match_descriptors`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors) function to find the features two images have in common. You'll get a list of pairs of indices of matched points.

Use the [`skimage.feature.plot_matches`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.plot_matches) function to plot the feature matches between the two images.

In [ ]:
# Your code here

### Solution

In [ ]:
aligned_kp = feature.match_descriptors(dsc1, dsc2)

In [ ]:
fig, ax = plt.subplots(dpi=200)
feature.plot_matches(ax, im, im_rotate, kp1, kp2, aligned_kp)
ax.axis("off")

## Aligning Consecutive Images

### Splitting the Image
Use the `split_in_two` function below to split your image in two. The two resulting images will share a section.

In [ ]:
def split_in_two(im: np.ndarray, split_value=None) -> tuple[np.ndarray, np.ndarray]:
  if not split_value:
    split_value = im.shape[1] // 4
  return im[:, :-split_value, :], im[:, split_value:, :]


# Your code here

#### Solution

In [ ]:
def split_in_two(im: np.ndarray, split_value=None) -> tuple[np.ndarray, np.ndarray]:
  if not split_value:
    split_value = im.shape[1] // 4
  return im[:, :-split_value, :], im[:, split_value:, :]


im_1, im_2 = split_in_two(
  im,
)
side_by_side((im_1, im_2))

### Extracting Keypoints and Descriptors
We'll now use the ORB feature extractor to find the features the images have in common, as in the previous exercise.

Store the keypoints and descriptors of each image and use the [`skimage.feature.match_descriptors`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_descriptors) function.

**Use the `max_ratio=0.8` parameter. It eliminates the least similar keypoints.**

The result of `match_descriptor` is the list of pairs of indices of matched keypoints.

In [ ]:
# Your code here

#### Solution

In [ ]:
feature_detector = ORB()
feature_detector.detect_and_extract(color.rgb2gray(im_1))
kp1 = feature_detector.keypoints
dsc1 = feature_detector.descriptors


feature_detector.detect_and_extract(color.rgb2gray(im_2))
kp2 = feature_detector.keypoints
dsc2 = feature_detector.descriptors

# Find the similar features between the two images
# Returns a list of index pairs (_,2)
aligned_kp = feature.match_descriptors(dsc1, dsc2, metric="hamming", max_ratio=0.8)

In [ ]:
from skimage import feature

fig, ax = plt.subplots(dpi=200)
feature.plot_matches(ax, im_1, im_2, kp1, kp2, aligned_kp)
ax.axis("off")

### Computing the Shared Area
 * Compute the average distance between the keypoint pairs
 * Compute the size of the area shared by the two images: $$zone\_size = image\_size - keypoint\_distance$$
 * Since the size of your shared area is a number of pixels, remember to turn it into an `int` with [`numpy.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html)

In [ ]:
# Your code here

#### Solution

In [ ]:
_, kp_dist = np.abs(kp2[aligned_kp[:, 1]] - kp1[aligned_kp[:, 0]]).mean(axis=0)
shared_size = (im_1.shape[1] - kp_dist).astype(int)

### Merging the Images
* Merge the two images. We'll do it simply:
  * Remove from the right image a left part the size of the shared area
  * Concatenate the first image and the rest of the second image with [`np.concatenate`](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html).
* Display the result

In [ ]:
# Your code here

#### Solution

In [ ]:
plt.figure(dpi=100)
im_fusion = np.concatenate((im_1, im_2[:, shared_size:, :]), axis=1)
plt.imshow(im_fusion)
plt.axis("off")
plt.show()

Merging by blending the two images rather than cutting

In [ ]:
def merging_fusion(
  im1: np.ndarray, im2: np.ndarray, merging_size: int, nb_pixel_fusion=30
) -> np.ndarray:
  im_1_left = im1[:, :-merging_size, :]
  im_1_right = im1[:, -merging_size:, :]
  im_2_left = im2[:, :merging_size]
  im_2_right = im2[:, merging_size:]
  merging_ratio = np.zeros(merging_size)
  half_point = merging_size // 2
  merging_ratio[half_point:] = 1
  merging_ratio[half_point - nb_pixel_fusion : half_point + nb_pixel_fusion] = (
    scipy.special.expit(np.linspace(-1, 1, num=nb_pixel_fusion * 2))
  )
  merging_ratio = np.expand_dims(merging_ratio, axis=(0, 2))
  im_center = im_1_right * (1 - merging_ratio) + im_2_left * merging_ratio
  return np.concatenate((im_1_left, im_center, im_2_right), axis=1).astype(np.uint8)


plt.figure(dpi=200)
plt.imshow(merging_fusion(im_1, im_2, shared_size))
plt.axis("off")

### Image Comparison Metrics
From your image:
* Convert your image to floating point values between 0 and 1 and store it in `im_float`. You can use [`skimage.img_as_float`](https://scikit-image.org/docs/stable/api/skimage.util.html#skimage.util.img_as_float) for that, or do it yourself
* Create a noisy image `im_noise` by adding a Gaussian noise image of the same size as your image, using [`np.random.normal`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.normal.html). You'll need [`numpy.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html#numpy.clip) to bring all values back between 0 and 1.
* Create an image `im_constant` that is your original image plus a constant. Use [`numpy.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html#numpy.clip) if needed
* Create an image `im_miroir` that is the mirror image along the horizontal axis

Then compare your `im_float` image with `im_noise`, `im_constant` and `im_miroir` using the metrics:
* [`skimage.metrics.structural_similarity`](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity): an image-specific metric meant to match our perception. Values in $[0,1]$; optimal at $1$
* [`skimage.metrics.mean_squared_error`](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.mean_squared_error): mean squared error in $[0, + \infty[$; optimal at $0$

In [ ]:
# Your code here
# im_float = ???
# im_noise = ???
# im_constant = ???
# im_miroir = ???

### Solution

In [ ]:
from skimage.metrics import mean_squared_error
from skimage.metrics import structural_similarity as ssim

noise = np.random.normal(0, 0.2, im.shape)

im_float = skimage.img_as_float(im)
mse_float = mean_squared_error(im_float, im_float)
ssim_float = ssim(im_float, im_float, channel_axis=2)

im_noise = np.clip(im_float + noise, 0, 1)
mse_noise = mean_squared_error(im_float, im_noise)
ssim_noise = ssim(im_float, im_noise, channel_axis=2)

im_constant = np.clip(im_float + 0.2, 0, 1)
mse_constant = mean_squared_error(im_float, im_constant)
ssim_constant = ssim(im_float, im_constant, channel_axis=2)

im_miroir = im_float[:, ::-1]
mse_miroir = mean_squared_error(im_float, im_miroir)
ssim_miroir = ssim(im_float, im_miroir, channel_axis=2)

# Measures
metric_float = f"MSE: {mse_float:.2f}, SSIM: {ssim_float:.2f}"
metric_noise = f"MSE: {mse_noise:.2f}, SSIM: {ssim_noise:.2f}"
metric_constant = f"MSE: {mse_constant:.2f}, SSIM: {ssim_constant:.2f}"
metric_miroir = f"MSE: {mse_miroir:.2f}, SSIM: {ssim_miroir:.2f}"


side_by_side(
  (im_float, im_noise, im_constant, im_miroir),
  subtitles=(metric_float, metric_noise, metric_constant, metric_miroir),
  dpi=200,
)

## Contouring

Contouring example with a support area

In [ ]:
from skimage.color import rgb2gray
from skimage.filters import gaussian
from skimage.segmentation import active_contour

POSITION_ELIPSE = (950, 770)
HAUTEUR_ELIPSE = 300
LARGEUR_ELIPSE = 400

s = np.linspace(0, 2 * np.pi, 400)
r = POSITION_ELIPSE[0] + HAUTEUR_ELIPSE * np.sin(s)
c = POSITION_ELIPSE[1] + LARGEUR_ELIPSE * np.cos(s)
init = np.array([r, c]).T

im_bw = (rgb2gray(im) * 255).astype(np.uint8)

snake = active_contour(
  gaussian(im_bw, 3),
  init,
  alpha=0.020,
)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(im, cmap=plt.cm.gray)
ax.plot(init[:, 1], init[:, 0], "--r", lw=3)
ax.plot(snake[:, 1], snake[:, 0], "-b", lw=3)
ax.set_xticks([]), ax.set_yticks([])
ax.axis([0, im.shape[1], im.shape[0], 0])

plt.show()